# PyCaret Time Series API for Bitcoin Forecasting

This notebook demonstrates real-time Bitcoin price forecasting using:
- Live data from CoinGecko API
- PyCaret's time series module
- Professional data pipeline practices

**References**:
- [CoinGecko API Docs](https://www.coingecko.com/en/api/documentation)
- [PyCaret Time Series Docs](https://pycaret.gitbook.io/docs/get-started/modules)



In [4]:
import os
os.environ['LD_PRELOAD'] = '/lib/aarch64-linux-gnu/libgomp.so.1'  # For ARM systems
# OR for regular Linux:
# os.environ['LD_PRELOAD'] = '/usr/lib/x86_64-linux-gnu/libgomp.so.1'

# Must be set BEFORE importing pycaret
from pycaret.time_series import *

In [5]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

# Imports
import logging
import requests
import numpy as np
import pandas as pd
import plotly.express as px
from pycaret.time_series import *
from datetime import datetime, timedelta

# Configuration
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)
pd.set_option('display.max_columns', 50)


In [6]:
class CoinGeckoAPI:
    """
    Professional API client for CoinGecko with error handling and rate limiting.
    """
    
    BASE_URL = "https://api.coingecko.com/api/v3"
    
    def __init__(self, api_key=None):
        self.session = requests.Session()
        self.api_key = api_key
        self.last_call = None
        self.RATE_LIMIT = 10  # calls per minute
        
    def get_ohlc(self, coin_id="bitcoin", vs_currency="usd", days="max"):
        """
        Get OHLC data with professional error handling.
        
        :param coin_id: Cryptocurrency ID (default: bitcoin)
        :param vs_currency: Target currency (default: usd)
        :param days: Data history length (default: max)
        :return: DataFrame with columns [timestamp, open, high, low, close]
        """
        endpoint = f"{self.BASE_URL}/coins/{coin_id}/ohlc"
        params = {
            "vs_currency": vs_currency,
            "days": days
        }
        
        try:
            response = self.session.get(endpoint, params=params)
            response.raise_for_status()
            data = response.json()
            
            df = pd.DataFrame(data, columns=['timestamp', 'open', 'high', 'low', 'close'])
            df['timestamp'] = pd.to_datetime(df['timestamp'], unit='ms')
            df.set_index('timestamp', inplace=True)
            
            logger.info(f"Successfully fetched {len(df)} records")
            return df
            
        except Exception as e:
            logger.error(f"API Error: {str(e)}")
            raise

# Initialize client
cg = CoinGeckoAPI()

In [8]:
def fetch_and_validate_data(days=90):
    """
    End-to-end data pipeline with quality checks and feature engineering.
    
    Parameters:
    -----------
    days : int
        Number of days of historical data to fetch
        
    Returns:
    --------
    pd.DataFrame
        Processed data ready for analysis
    """
    # Fetch raw data
    raw_data = cg.get_ohlc(days=days)
    
    # Validation Checks
    assert isinstance(raw_data, pd.DataFrame), "Data must be a DataFrame"
    assert not raw_data.empty, "Empty data received"
    assert {'open', 'high', 'low', 'close'}.issubset(raw_data.columns), "Missing OHLC columns"
    
    # Data Cleaning
    data = raw_data.copy()
    data = data[~data.index.duplicated(keep='first')]
    
    # Feature Engineering
    data['daily_return'] = data['close'].pct_change()
    data['volatility_7d'] = data['daily_return'].rolling(7).std()
    data['volume_ema_14'] = data['close'].ewm(span=14).mean()
    
    logger.info(f"Data validation passed. Final shape: {data.shape}")
    return data

# Fetch and display data
btc_data = fetch_and_validate_data()
display(btc_data.tail(3))

ERROR:__main__:API Error: HTTPSConnectionPool(host='api.coingecko.com', port=443): Max retries exceeded with url: /api/v3/coins/bitcoin/ohlc?vs_currency=usd&days=90 (Caused by NewConnectionError('<urllib3.connection.VerifiedHTTPSConnection object at 0xffff3cedc460>: Failed to establish a new connection: [Errno -2] Name or service not known'))


ConnectionError: HTTPSConnectionPool(host='api.coingecko.com', port=443): Max retries exceeded with url: /api/v3/coins/bitcoin/ohlc?vs_currency=usd&days=90 (Caused by NewConnectionError('<urllib3.connection.VerifiedHTTPSConnection object at 0xffff3cedc460>: Failed to establish a new connection: [Errno -2] Name or service not known'))

In [9]:
def setup_experiment(data):
    """
    Properly configured PyCaret time series experiment for financial data.
    
    Parameters:
    -----------
    data : pd.DataFrame
        Must have datetime index and numeric columns
        
    Returns:
    --------
    TSForecastingExperiment
    """
    try:
        # Ensure proper data formatting
        if not isinstance(data.index, pd.DatetimeIndex):
            raise ValueError("Data must have datetime index")
            
        # Convert to numeric and drop NA
        numeric_data = data.select_dtypes(include=np.number)
        numeric_data = numeric_data.dropna()
        
        # Initialize experiment
        exp = setup(
            data=numeric_data,
            target='close',
            index=data.index,  # Explicitly pass datetime index
            fold_strategy='expanding',
            fold=3,
            numeric_imputation_target='ffill',
            session_id=42,
            fh=7,
            verbose=True  # Enable for debugging
        )
        
        logger.info("PyCaret experiment successfully initialized")
        return exp
        
    except Exception as e:
        logger.error(f"Setup failed: {str(e)}")
        raise

# Properly formatted data
btc_data_formatted = btc_data[['close']].copy()
btc_data_formatted = btc_data_formatted.asfreq('D').ffill()  # Ensure daily frequency

# Initialize experiment
exp = setup_experiment(btc_data_formatted)

NameError: name 'btc_data' is not defined

In [13]:
print(type(btc_data.index))  # Should show <class 'pandas.core.indexes.datetimes.DatetimeIndex'>


<class 'pandas.core.indexes.datetimes.DatetimeIndex'>


In [14]:
print(btc_data.dtypes)  # All columns should be numeric

open             float64
high             float64
low              float64
close            float64
daily_return     float64
volatility_7d    float64
volume_ema_14    float64
dtype: object


In [15]:
print(btc_data.index.freq)  # Should show frequency (e.g. 'D' for daily)

None


In [16]:
import pandas as pd
from pycaret.time_series import *

# Convert index to DatetimeIndex if needed
btc_data.index = pd.to_datetime(btc_data.index)

# Set explicit frequency (critical for PyCaret)
btc_data = btc_data.asfreq('D')  # Daily frequency
btc_data = btc_data.ffill()      # Forward fill missing values

# Verify frequency is set
print(f"Frequency: {btc_data.index.freq}")  # Should now show 'D'

Frequency: <Day>


In [3]:
def robust_pycaret_setup(data):
    """
    PyCaret setup for financial time series.
    
    """
    # 1. Ensure proper data structure
    if not isinstance(data, pd.DataFrame):
        data = pd.DataFrame(data)
    
    # 2. Validate and clean
    assert 'close' in data.columns, "Data must contain 'close' column"
    data = data[['close']].copy()
    data = data.dropna()
    
    # 3. Force daily frequency
    data = data.asfreq('D')
    data = data.ffill()
    
    # 4. Initialize experiment
    exp = setup(
        data=data,
        target='close',
        fold_strategy='expanding',
        fold=3,
        numeric_imputation_target='ffill',
        session_id=42,
        fh=7,  # 7-period forecast
        verbose=True
    )
    return exp

# Run with your data
exp = robust_pycaret_setup(btc_data)

NameError: name 'btc_data' is not defined